# Track AI Agent Costs

AI agents can burn through tokens fast. A single Claude Code session with subagents can use 100k+ tokens across dozens of API calls. This notebook shows how to track what those calls actually cost, from a single response to an entire project.

We'll cover:
1. Estimating cost from any API response
2. Comparing costs across models
3. Tracking costs across a session from JSONL transcripts
4. Analyzing cache savings

## Setup

Install the [LLMKit SDK](https://github.com/smigolsmigol/llmkit), which includes a bundled pricing table for common LLM models.

In [1]:
%pip install llmkit-sdk anthropic python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


C:\f3d1\claude-cookbooks\.venv\Scripts\python.exe: No module named pip


## 1. Estimate cost from a single response

The simplest approach: wrap your HTTP client with `tracked()`. It intercepts responses and estimates costs from the `usage` field using a bundled pricing table. No API key, no server, no config.

In [2]:
from dotenv import load_dotenv

load_dotenv()

import anthropic
from llmkit import tracked

# tracked() returns an httpx.Client that estimates costs automatically
client = anthropic.Anthropic(http_client=tracked())

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)

print(response.content[0].text)
print(f"\nTokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out")

The capital of France is Paris.

Tokens: 14 in, 10 out


## 2. Comparing costs across models

The same prompt costs very different amounts depending on the model. Use `calculate_cost()` to compare before committing to a model for a workload.

In [3]:
from llmkit import calculate_cost

# simulate a typical agent workload: 2k input, 1k output per call, 50 calls
calls = 50
input_per_call = 2000
output_per_call = 1000

models = [
    "claude-haiku-4-5",
    "claude-sonnet-4-6",
    "claude-opus-4-6",
    "gpt-4o",
    "gpt-4.1-mini",
    "gemini-2.0-flash",
    "deepseek-chat",
]

print(f"Cost comparison: {calls} calls x {input_per_call} in + {output_per_call} out per call\n")
print(f"{'Model':<25} {'Per call':>10} {'Total ({calls} calls)':>18}")
print("-" * 55)

for model in models:
    per_call = calculate_cost(model, input_per_call, output_per_call)
    if per_call is not None:
        total = per_call * calls
        print(f"{model:<25} ${per_call:>9.4f} ${total:>17.2f}")
    else:
        print(f"{model:<25} {'(not priced)':>10}")

Cost comparison: 50 calls x 2000 in + 1000 out per call

Model                       Per call Total ({calls} calls)
-------------------------------------------------------
claude-haiku-4-5          $   0.0070 $             0.35
claude-sonnet-4-6         $   0.0210 $             1.05
claude-opus-4-6           $   0.0350 $             1.75
gpt-4o                    $   0.0150 $             0.75
gpt-4.1-mini              $   0.0024 $             0.12
gemini-2.0-flash          $   0.0006 $             0.03
deepseek-chat             $   0.0010 $             0.05


## 3. Track session costs from transcripts

Claude Code stores conversation transcripts as JSONL files in `~/.claude/projects/`. Each assistant message includes token usage. Let's parse a session transcript and calculate the total cost.

We'll use a sample transcript included with this notebook (the same format Claude Code writes).

In [4]:
import json
from pathlib import Path

from llmkit import calculate_cost

# parse a Claude Code session transcript
transcript = Path("observability/track_agent_costs/sample_session.jsonl")
if not transcript.exists():
    transcript = Path("sample_session.jsonl")  # fallback for local runs

total_cost = 0.0
total_input = 0
total_output = 0
total_cache_read = 0
total_cache_write = 0
messages = 0

for line in transcript.read_text().strip().split("\n"):
    data = json.loads(line)
    if data.get("type") != "assistant":
        continue

    msg = data["message"]
    usage = msg.get("usage", {})
    model = msg.get("model", "")

    inp = usage.get("input_tokens", 0)
    out = usage.get("output_tokens", 0)
    cache_read = usage.get("cache_read_input_tokens", 0)
    cache_write = usage.get("cache_creation_input_tokens", 0)

    cost = calculate_cost(model, inp, out, cache_read, cache_write)
    if cost is not None:
        total_cost += cost

    total_input += inp
    total_output += out
    total_cache_read += cache_read
    total_cache_write += cache_write
    messages += 1

print(f"Session: {messages} messages")
print(f"Tokens: {total_input:,} input, {total_output:,} output")
print(f"Cache: {total_cache_read:,} read, {total_cache_write:,} write")
print(f"Estimated cost: ${total_cost:.4f}")

Session: 5 messages
Tokens: 5,460 input, 6,530 output
Cache: 49,400 read, 4,750 write
Estimated cost: $0.1470


## 4. Analyze cache savings

Prompt caching can save significant money. Cache reads cost ~10% of regular input tokens. Let's see how much caching saved in this session.

In [5]:
from llmkit import calculate_cost

# compare what cache reads cost vs what they would cost at full price
# claude-sonnet-4-6: $3/M input, $0.30/M cache read (90% savings)
INPUT_PRICE_PER_M = 3.0
CACHE_READ_PRICE_PER_M = 0.30

# what cache reads WOULD have cost at full input price
full_price = (total_cache_read / 1_000_000) * INPUT_PRICE_PER_M
# what they actually cost at cache read price
cache_price = (total_cache_read / 1_000_000) * CACHE_READ_PRICE_PER_M
savings = full_price - cache_price

print(f"Cache read tokens: {total_cache_read:,}")
print(f"Full price would be: ${full_price:.4f}")
print(f"Cache price was: ${cache_price:.4f}")
print(f"Savings: ${savings:.4f} ({savings / full_price * 100:.0f}% reduction)")

if total_cache_write > 0:
    ratio = total_cache_read / total_cache_write
    print(f"\nCache efficiency: {ratio:.1f}x read-to-write ratio")
    print("(Higher is better. Above 3x means caching is paying for itself)")

Cache read tokens: 49,400
Full price would be: $0.1482
Cache price was: $0.0148
Savings: $0.1334 (90% reduction)

Cache efficiency: 10.4x read-to-write ratio
(Higher is better. Above 3x means caching is paying for itself)


## Next steps

This notebook covered client-side cost estimation: per-response pricing, cross-model comparison, session-level tracking from transcripts, and cache savings analysis. All of this runs locally with no API keys beyond the one you already use.

For real-time cost visibility inside Claude Code, Cursor, or Cline, the [LLMKit MCP server](https://github.com/smigolsmigol/llmkit/tree/main/packages/mcp-server) provides tools that surface this data directly in your editor. For server-side budget enforcement, see the [LLMKit proxy](https://github.com/smigolsmigol/llmkit).